# News Pipeline — Step by Step

Agência Brasil dengue HTML → frozen text corpus → weekly per-UF lexicon features,
one step at a time.

This notebook mirrors `src/pipeline/news/` exactly: it unrolls `build()` so each
transformation is visible. It runs on the committed fixtures in
`tests/fixtures/news/` — five saved pages, one per extraction path — so it
executes instantly. The last three sections summarise the real corpus, the real
features, and join them to SINAN when those artefacts are on disk.

**Pipeline:** manifest + HTML → era-aware extract → `ew` (Sunday) → frozen corpus
→ per-article lexicon signals → (ew, location) features → optional SINAN join

## How to read this notebook

The project compares **three competing representations of one news corpus** —
three ways of turning the same articles into model features — and asks which one
predicts dengue best. Each representation is called a **braço** (Portuguese for
"arm", as in the arms of a trial):

| Braço | Representation | Status |
|---|---|---|
| **01** | PT-BR lexicon + hand-written rules — **this notebook** | implemented |
| 02 | dense embeddings, pooled per epi-week | schema only, refuses to run |
| 03 | temporal topic modelling | schema only, refuses to run |

Two further paths — braço 04 (structured extraction with an LLM) and braço 05 (an
LLM forecaster) — reuse the same corpus but produce a different kind of output.
They exist as research notes only, with no code in this pipeline.

For the comparison to mean anything, everything except the representation has to
be held fixed: same corpus, same calendar, same target, same model, same metrics.
Those fixed pieces are written down in
[`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md), which
the project calls **protocol 00**. The open questions behind braço 01 — including
the annotation study this notebook does **not** run — are in
[`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md).

Three more terms recur throughout:

- **Era B / era C** — Agência Brasil redesigned its page template around 2014, so
  the corpus holds two generations of HTML that need different extraction rules.
  Era B is 2002–2014 and puts the body in `div.inside`; era C is 2014–2026 and
  uses `div.node-conteudo`. Step 2 handles the split.
- **`ew`** — epidemiological week, identified by the **Sunday** that starts it.
  Every source in this project uses the same Sunday calendar, so `ew` is the join
  key across news, SINAN, Google Trends and climate.
- **Rail** — the "Últimas notícias" (latest news) sidebar that appears on era-B
  pages. It is the trap Step 3 is about.

**Literature / project alignment:**

| Choice | Grounding |
|---|---|
| The text corpus is its own stage | All three braços must read one identical input, or the comparison measures the parser instead of the representation |
| Dictionary + rules, no model | HealthMap / BioCaster / PADI-web lineage; the only dengue-specific precedent with peer-reviewed timing evidence |
| `news_volume` counts body-less pages | The article was published; only its text is missing |
| Max, not sum, for weekly counts | Counts in this corpus are cumulative-to-date, so a sum would scale with how often the press repeated the same figure |
| PT-BR filter on the lexicon | 171 of 4,569 articles are Spanish, 61 of them from 2016 |
| National news, UF target — declared, not solved | The spatial mismatch belongs to the source, not to this stage |


## Setup

In [1]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.news.common import strip_accents
from pipeline.news.text.spec import (
    ARTICLE_ID, CHAR_COUNT, ERA, EW, EXTRACTOR, LANG_GUESS, MIN_CHARS,
    TEXT, TITLE,
)
from pipeline.news.text.spec import OUTPUT_COLUMNS as CORPUS_COLUMNS
from pipeline.news.text import steps as text_steps
from pipeline.news.text.integrity import run_all as corpus_checks
from pipeline.news.lexicon.spec import (
    LOCATION, NEWS_ARTICLES_WITH_CASES, NEWS_CASES, NEWS_DEATHS,
    NEWS_SEVERITY, NEWS_VOLUME,
)
from pipeline.news.lexicon.spec import OUTPUT_COLUMNS as FEATURE_COLUMNS
from pipeline.news.lexicon import steps as lex_steps
from pipeline.news.lexicon.integrity import run_all as feature_checks

pl.Config.set_tbl_rows(14)
pl.Config.set_fmt_str_lengths(70)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "news"
LEXICON_PATH = PROJECT_ROOT / "data" / "news" / "reference" / "lexico_dengue.yaml"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"

CORPUS_PATH = PROJECT_ROOT / "data" / "pipeline" / "news" / "EBC_dengue_text.parquet"
FEATURES_PATH = PROJECT_ROOT / "data" / "pipeline" / "news" / "EBC_dengue_lexicon_EW.parquet"
SINAN_PATH = PROJECT_ROOT / "data" / "epidemiological" / "SINAN" / "SINAN_dengue_cases.parquet"

FIXTURES.exists(), LEXICON_PATH.exists(), FU_PATH.exists()

(True, True, True)

## Literature checklist (what this notebook must prove)

1. Both template generations extract a **real body** — era B from `div.inside`,
   era C from `div.node-conteudo` — and no era-B body is the "Últimas notícias"
   rail.
2. `ew` is always a **Sunday**, and it is the same calendar SINAN is keyed on.
3. `char_count` is `len(text)`, and the shortest real article clears `MIN_CHARS`.
4. Every article contributes a `BR` row; only some also name a UF.
5. A week with no article is **absent**, not zero.
6. `news_cases` / `news_deaths` are **maxima**, not sums, and are null in most
   weeks.
7. The fixture is only five pages, so it cannot show the Spanish slice, the count
   gates at scale, or the per-UF bias. Those need the real corpus, and the
   **Optional — real** sections below are where they are measured.


## Step 1 — Manifest + HTML → rows

`load_manifest` reads the scraper's `manifest.jsonl` and keeps only entries with a
saved file — the scraper records every URL it saw, including two whose fetch
failed. `iter_rows` then extracts each page into a dict.

In [2]:
manifest = text_steps.load_manifest(FIXTURES)
print(f"{len(manifest)} manifest rows carry a saved article")

rows = list(text_steps.iter_rows(FIXTURES))
{ARTICLE_ID: rows[0][ARTICLE_ID], ERA: rows[0][ERA], CHAR_COUNT: rows[0][CHAR_COUNT]}

5 manifest rows carry a saved article


{'article_id': '2002-09-30_584838_b533c1', 'era': 'b', 'char_count': 231}

## Step 2 — Two template generations, one extractor each

The site was redesigned at ~2014 and both generations are in the corpus. Era B
puts the body in `div.inside` and the *site name* in `<h1>`; era C uses
`div.node-conteudo` and the real title in `<h1>`. Trusting `<h1>` would have
labelled 2,628 articles "Agência Brasil".

`div.inside` matches **three** elements per era-B page — a 20-character header,
the article, and a news rail — so the extractor scores candidates rather than
taking the first. The score is `(paragraph text, link-free text)`: an article has
`<p>` elements and a rail does not.

In [3]:
corpus = text_steps.build(FIXTURES)
assert tuple(corpus.columns) == CORPUS_COLUMNS
corpus.group_by(ERA, EXTRACTOR).len().sort(ERA, EXTRACTOR)

era,extractor,len
str,str,u32
"""b""","""div.inside""",3
"""c""","""div.node-conteudo""",1
"""fallback""","""largest-paragraph-block""",1


## Step 3 — What the rail trap looked like

This is the more dangerous of the two extraction bugs the project names. **Bug 1**
took a 20-character page header as the body; **bug 2** took the "Últimas notícias"
sidebar instead of the article. Step 5 shows how the length checks tell them
apart.

Bug 2 is the one that could have shipped silently. Scoring candidates on *total*
text picked the rail for **1,097 of the 2,628 era-B articles (42%)** — 1,086
characters of link text against a 677-character article, long enough to pass every
length check. The fix scores on paragraph text first, which the rail lacks.

The three era-B fixture rows below are the shortest in the corpus, which is where
a regression would show. The first is a **body-less page**: the fetch saved the
headline and then jumped straight to the rail, so all that is left is the byline.


In [4]:
(
    corpus.filter(pl.col(ERA) == "b")
    .sort(CHAR_COUNT)
    .select(ARTICLE_ID, EXTRACTOR, CHAR_COUNT,
            pl.col(TEXT).str.slice(0, 70).alias("text_head"))
)

article_id,extractor,char_count,text_head
str,str,i64,str
"""2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre…","""div.inside""",40,"""Danilo Macedo Repórter da Agência Brasil"""
"""2002-09-30_584838_b533c1""","""div.inside""",231,"""Brasília, 30 (Agência Brasil - ABr) - O 1º Fórum Artístico de Combate """
"""2011-01-11_ministros-discutem-acoes-de-prevencao-e-controle-da-dengue-…","""div.inside""",539,"""Da Agência Brasil Brasília - Será realizada hoje (11), às 9h, no Palá"""


## Step 4 — `ew`: the Sunday that starts the epi week

`add_ew` parses `published_at` and floors it to Sunday with the shared
`pipeline.epiweek.floor_to_sunday` — never `dt.truncate("1w")`, which polars
anchors on Monday. SINAN, Google Trends and climate all use the same helper, so
the join axis is one definition rather than four.

In [5]:
dated = text_steps.add_ew(pl.DataFrame(rows, schema=text_steps.SCHEMA))
assert dated.filter(pl.col(EW).dt.weekday() != 7).is_empty(), "not all Sundays"
dated.select(ARTICLE_ID, "published_at", EW).sort(EW)

article_id,published_at,ew
str,datetime[μs],date
"""2002-09-30_584838_b533c1""",2002-09-30 08:14:00,2002-09-29
"""2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre…",2008-04-02 18:16:00,2008-03-30
"""2011-01-11_ministros-discutem-acoes-de-prevencao-e-controle-da-dengue-…",2011-01-11 06:14:00,2011-01-09
"""2016-02-04_presidente-dilma-pede-uniao-no-combate-ao-aedes-aegypti_1ce…",2016-02-04 13:48:00,2016-01-31
"""2021-03-14_synthetic-fallback-page_000001""",2021-03-14 09:00:00,2021-03-14


## Step 5 — `char_count` and the length floor

`char_count` is `len(text)`, stored so a consumer can filter without touching the
body. `MIN_CHARS` is 100: a correct extraction bottoms out at 128 characters (a
one-sentence 2002 wire note), while a header-only regression yields about 20.

`bodyless_expr` separates the two cases. It is anchored on the byline's trailing
site credit rather than on shortness alone, and that anchor is what makes it work:
bug 1 also produced 20-character bodies, so a length-only rule would reclassify
that regression as a truncated fetch and exempt it from `min_length`. The check
would stop checking.


In [6]:
(
    corpus.with_columns(text_steps.bodyless_expr().alias("bodyless"))
    .select(ERA, CHAR_COUNT, "bodyless",
            pl.col(TEXT).str.slice(0, 45).alias("text_head"))
    .sort(CHAR_COUNT)
)

era,char_count,bodyless,text_head
str,i64,bool,str
"""b""",40,true,"""Danilo Macedo Repórter da Agência Brasil"""
"""c""",163,false,"""Em cadeia nacional de rádio e TV a presidenta"""
"""b""",231,false,"""Brasília, 30 (Agência Brasil - ABr) - O 1º Fó"""
"""b""",539,false,"""Da Agência Brasil Brasília - Será realizada """
"""fallback""",583,false,"""A Secretaria Municipal de Saúde ampliou nesta"""


## Step 6 — `build()` chains it: the frozen corpus

`build()` = rows → declared schema → `add_ew` → `order_columns`. The published
column order is fixed, and the schema is declared rather than inferred so an
all-null column still gets its published dtype.

In [7]:
corpus.select(ARTICLE_ID, EW, ERA, EXTRACTOR, CHAR_COUNT, LANG_GUESS)

article_id,ew,era,extractor,char_count,lang_guess
str,date,str,str,i64,str
"""2002-09-30_584838_b533c1""",2002-09-29,"""b""","""div.inside""",231,"""pt"""
"""2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre…",2008-03-30,"""b""","""div.inside""",40,"""pt"""
"""2011-01-11_ministros-discutem-acoes-de-prevencao-e-controle-da-dengue-…",2011-01-09,"""b""","""div.inside""",539,"""pt"""
"""2016-02-04_presidente-dilma-pede-uniao-no-combate-ao-aedes-aegypti_1ce…",2016-01-31,"""c""","""div.node-conteudo""",163,"""pt"""
"""2021-03-14_synthetic-fallback-page_000001""",2021-03-14,"""fallback""","""largest-paragraph-block""",583,"""pt"""


## Integrity checks — corpus

The same ten checks the CLI and the test suite run. Two of them **report rather
than fail**, and that distinction is deliberate: `langs_known` and
`bodyless_pages` describe properties of the *source* (Agência Brasil publishes in
Spanish; three pages were fetched without a body), while `min_length` and
`title_agreement` describe defects in *this stage* and do fail.

In [8]:
for check in corpus_checks(corpus):
    print(check)

[PASS] schema: 12 columns in published order
[PASS] unique_keys: 5 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[PASS] min_length: 5 rows, shortest 40 chars
[PASS] bodyless_pages: 1 page(s) saved without a body (e.g. 2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre-dengue-diz-temporao_861d4e)
[PASS] eras_known: b=3, c=1, fallback=1 — 1 via heuristic
[PASS] langs_known: pt=5
[PASS] text_present: 5 non-empty bodies
[PASS] title_present: 5 titles present
[PASS] title_agreement: 100.0% of 5 rows agree


## Step 7 — Braço 01: the lexicon and the UF matcher

The lexicon is versioned YAML. Terms are stored accent-folded and matched against
a normalised copy of the text, so `obito` matches `óbito`; one-word terms get
their plural generated (`-cao → -coes`, `-r/-z → +es`). Severity carries ordinal
weights — `alert` 1, `outbreak` 2, `epidemic` 3.

Full state names match **case-sensitively on the original text**, and that matters
more than it looks: accent-folding would make `Pará` and the preposition `para`
the same string, scoping nearly every article to Pará. Two-letter codes need a
locative cue (`no estado de SP`), because `SE` is a reflexive pronoun and `PA`
collides with `p.a.`


In [9]:
lexicon = lex_steps.load_lexicon(LEXICON_PATH)
matcher = lex_steps.load_uf_matcher(FU_PATH)
{
    "version": lexicon.version,
    "severity_tiers": {tier.name: tier.weight for tier in lexicon.severity},
    "disease_terms": len(lexicon.disease),
    "clinical_terms": len(lexicon.clinical),
    "vector_terms": len(lexicon.vector),
    "uf_name_patterns": len(matcher.name_patterns),
    "require_disease_term": lexicon.require_disease_term,
}

{'version': 1,
 'severity_tiers': {'epidemic': 3, 'outbreak': 2, 'alert': 1},
 'disease_terms': 8,
 'clinical_terms': 23,
 'vector_terms': 11,
 'uf_name_patterns': 27,
 'require_disease_term': True}

## Step 8 — Per-article signals

`analyse` folds title and body together, accent-folds once, and returns the
severity score, the largest case and death count that survived the gates, whether
a disease term is present, and which UFs were named.

On the fixture every severity is **0**: none of the five saved pages uses
"surto", "epidemia" or "alerta". That is a property of the sample, not of the
scorer — the real corpus is where severity has signal (median 4 per week at
national level).

In [10]:
for row in corpus.filter(pl.col(CHAR_COUNT) > 100).sort(EW).iter_rows(named=True):
    signals = lex_steps.analyse(row[TEXT], row[TITLE], lexicon, matcher)
    print(
        f"{row[ARTICLE_ID][:34]:34}  severity={signals.severity:2}"
        f"  cases={signals.cases}  deaths={signals.deaths}"
        f"  ufs={sorted(signals.locations)}"
    )

2002-09-30_584838_b533c1            severity= 0  cases=None  deaths=None  ufs=['RJ']
2011-01-11_ministros-discutem-acoe  severity= 0  cases=None  deaths=None  ufs=[]
2016-02-04_presidente-dilma-pede-u  severity= 0  cases=None  deaths=None  ufs=[]
2021-03-14_synthetic-fallback-page  severity= 0  cases=1240  deaths=None  ufs=[]


## Step 9 — Aggregate to (ew, location)

Only observed pairs are emitted. A week with no article is **absent**, not zero:
emitting the dense grid would assert "zero coverage" for every UF that national
news simply never named, which is a different claim from "no coverage this week".
The consumer left-joins onto the target calendar.

`news_volume` is a count, `news_severity` a sum, and the two count slots a
**max**.

In [11]:
features = lex_steps.build(corpus, lexicon, matcher)
assert tuple(features.columns) == FEATURE_COLUMNS
features

ew,location,news_volume,news_severity,news_cases,news_deaths,news_articles_with_cases
date,str,u32,i64,i64,i64,i64
2002-09-29,"""BR""",1,0,null,null,0
2002-09-29,"""RJ""",1,0,null,null,0
2008-03-30,"""BR""",1,0,null,null,0
2011-01-09,"""BR""",1,0,null,null,0
2016-01-31,"""BR""",1,0,null,null,0
2021-03-14,"""BR""",1,0,1240,null,1


## Step 10 — The three gates on counts

Raw number extraction from news prose is dominated by numbers that are not about
dengue. A count is kept only if it is:

1. **not negated** — no negation cue in the preceding 40 characters;
2. **not qualified by another disease** — `11.881.430 casos de DRSAI` is not ours;
3. **in a sentence that names a dengue-family disease**.

Gate 3 is the strictest and the one that costs recall: `No estado de São Paulo,
foram 36.791 casos` is dropped when the sentence never says "dengue". That
direction is deliberate — a count an order of magnitude wrong is worse for the
model than a count that is missing.

Two rules that are not obvious follow the probe. `mil` multiplies only when the
number is *not* already dotted-thousands-grouped (`36.791 mil casos` is 36,791,
not 36.8 million). And severity is ordinal: the last probe line scores 6, because
"alerta" + "surto" + "epidemia" are 1 + 2 + 3.


In [12]:
probe = [
    "São Paulo registrou 36.791 casos de dengue em 2024.",
    "No estado de São Paulo, foram 36.791 casos no ano passado.",
    "Não houve 500 casos de dengue na cidade.",
    "Foram 11.881.430 casos de DRSAI no país.",
    "O país somou 700 mil mortes por covid, segundo estudo sobre dengue.",
    "Dengue: 500 mil casos prováveis e 312 óbitos.",
    "Dengue: 36.791 mil casos.",
    "Alerta para o surto de dengue; pode haver epidemia.",
]
for line in probe:
    normalized = strip_accents(line)
    cases, deaths = lex_steps.count_mentions(normalized, lexicon)
    severity = lex_steps.severity_score(normalized, lexicon)
    print(f"{line[:56]:58} cases={str(cases):7} deaths={str(deaths):5} severity={severity}")

São Paulo registrou 36.791 casos de dengue em 2024.        cases=36791   deaths=None  severity=0
No estado de São Paulo, foram 36.791 casos no ano passad   cases=None    deaths=None  severity=0
Não houve 500 casos de dengue na cidade.                   cases=None    deaths=None  severity=0
Foram 11.881.430 casos de DRSAI no país.                   cases=None    deaths=None  severity=0
O país somou 700 mil mortes por covid, segundo estudo so   cases=None    deaths=None  severity=0
Dengue: 500 mil casos prováveis e 312 óbitos.              cases=500000  deaths=312   severity=0
Dengue: 36.791 mil casos.                                  cases=36791   deaths=None  severity=0
Alerta para o surto de dengue; pode haver epidemia.        cases=None    deaths=None  severity=6


## Integrity checks — features

Eight checks: schema, unique keys, Sundays, `locations ⊆ {BR} ∪ 27 UFs`,
non-negative counts, the cases domain, `articles_with_cases ≤ volume`, and **a BR
row in every week**. Every article contributes to BR by construction, so a week
without one means the location spread dropped something.

On the fixture, `locations_known` fails: five articles name only RJ, so 26 UFs
never appear. On the real output all 27 pass.

In [13]:
for check in feature_checks(features, FU_PATH):
    print(check)

[PASS] schema: 7 columns in published order
[PASS] unique_keys: 6 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[PASS] locations_known: 2 known locations
[PASS] non_negative: volume >= 1 and derived counts >= 0
[PASS] cases_domain: counts are null or positive
[PASS] articles_with_cases: contributors never exceed article volume
[PASS] national_row_per_week: 5 weeks all carry BR


## Optional — the real corpus

Requires `uv run arboili ebc` then `uv run arboili transform news-text` (~45 s
over 4,569 articles, 2002–2026).

In [14]:
if CORPUS_PATH.exists():
    real = pl.read_parquet(CORPUS_PATH)
    print(f"articles={real.height:,}   ew={real[EW].min()} → {real[EW].max()}")
    print(f"weeks={real[EW].n_unique():,}   shortest={real[CHAR_COUNT].min()} chars\n")
    print(real.group_by(ERA, EXTRACTOR).len().sort("len", descending=True))
    print(real.group_by(LANG_GUESS).len().sort(LANG_GUESS))
    print(real.select(CHAR_COUNT).describe())
else:
    print("Not built — run: uv run arboili transform news-text")

articles=4,569   ew=2002-02-17 → 2026-09-13
weeks=973   shortest=40 chars

shape: (2, 3)
┌─────┬───────────────────┬──────┐
│ era ┆ extractor         ┆ len  │
│ --- ┆ ---               ┆ ---  │
│ str ┆ str               ┆ u32  │
╞═════╪═══════════════════╪══════╡
│ b   ┆ div.inside        ┆ 2628 │
│ c   ┆ div.node-conteudo ┆ 1941 │
└─────┴───────────────────┴──────┘
shape: (2, 2)
┌────────────┬──────┐
│ lang_guess ┆ len  │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ other      ┆ 171  │
│ pt         ┆ 4398 │
└────────────┴──────┘
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ char_count  │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 4569.0      │
│ null_count ┆ 0.0         │
│ mean       ┆ 2201.450646 │
│ std        ┆ 1945.465363 │
│ min        ┆ 40.0        │
│ 25%        ┆ 1142.0      │
│ 50%        ┆ 1829.0      │
│ 75%        ┆ 2695.0      │
│ max        ┆ 34960.0     │
└────────────┴───────

## Optional — the real features

This is the measurement that answers the open question in
[`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md) about whether the two
count slots are worth keeping. Watch the `null_count` row: `news_volume` and
`news_severity` are present in **every** week, `news_cases` in about half, and
`news_deaths` in a fifth. That is why the notes route structured count extraction
to braço 04 (LLM extraction) instead of to this regex-based braço.


In [15]:
if FEATURES_PATH.exists():
    real_features = pl.read_parquet(FEATURES_PATH)
    national = real_features.filter(pl.col(LOCATION) == "BR")
    print(f"rows={real_features.height:,}  weeks={real_features[EW].n_unique():,}"
          f"  locations={real_features[LOCATION].n_unique()}\n")
    print(national.select(NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES, NEWS_DEATHS).describe())
    print()
    print(
        real_features.filter(pl.col(LOCATION) != "BR")
        .group_by(LOCATION).len().sort("len", descending=True).head(5)
        .rename({"len": "weeks_covered"})
    )
    print("weeks covered per UF — not articles: an article that names two states"
          " contributes a row to each, and the README's article counts (RJ 1,636"
          " vs SP 863) are a different measure")
else:
    print("Not built — run: uv run arboili transform news-lexicon")

rows=5,152  weeks=965  locations=28

shape: (9, 5)
┌────────────┬─────────────┬───────────────┬───────────────┬──────────────┐
│ statistic  ┆ news_volume ┆ news_severity ┆ news_cases    ┆ news_deaths  │
│ ---        ┆ ---         ┆ ---           ┆ ---           ┆ ---          │
│ str        ┆ f64         ┆ f64           ┆ f64           ┆ f64          │
╞════════════╪═════════════╪═══════════════╪═══════════════╪══════════════╡
│ count      ┆ 965.0       ┆ 965.0         ┆ 511.0         ┆ 189.0        │
│ null_count ┆ 0.0         ┆ 0.0           ┆ 454.0         ┆ 776.0        │
│ mean       ┆ 4.527461    ┆ 9.872539      ┆ 358237.64775  ┆ 3886.116402  │
│ std        ┆ 7.270023    ┆ 16.038443     ┆ 951118.116052 ┆ 37349.595202 │
│ min        ┆ 1.0         ┆ 0.0           ┆ 12.0          ┆ 1.0          │
│ 25%        ┆ 1.0         ┆ 0.0           ┆ 6437.0        ┆ 33.0         │
│ 50%        ┆ 2.0         ┆ 4.0           ┆ 59696.0       ┆ 91.0         │
│ 75%        ┆ 5.0         ┆ 13.0    

## Exploratory — does news volume track SINAN?

**This is not the protocol-00 comparison.** It is a first look at the paper's
actual question, and it should be read with three caveats attached.

`news_volume` is the defensible feature. `news_cases` is a **cumulative-to-date**
figure the press repeats, so correlating it with weekly incidence measures
something else entirely — it is in the table to show that, not to be used.
Coverage is sparse per UF: some states are named in only 32 of the 481 weeks the
two series share, which makes their coefficient noisy. And the news series is
national while the target is UF, so a per-UF correlation is really "does national
coverage move with this state's curve".

SINAN here is `ew_symptom_onset` — the same Sunday calendar — summed to
(week, state).


In [16]:
sinan_weekly = (
    pl.scan_parquet(SINAN_PATH)
    .group_by("ew_symptom_onset", "state_abbrev")
    .agg(pl.col("case_count").sum().alias("casos"))
    .collect()
)

if FEATURES_PATH.exists():
    br = (
        real_features.filter(pl.col(LOCATION) == "BR")
        .join(
            sinan_weekly.group_by("ew_symptom_onset")
            .agg(pl.col("casos").sum())
            .rename({"ew_symptom_onset": EW}),
            on=EW,
            how="inner",
        )
        .sort(EW)
    )
    print(f"national weeks with both series: {br.height:,}  "
          f"({br[EW].min()} → {br[EW].max()})\n")
    print(
        pl.DataFrame({
            "feature": [NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES],
            "spearman vs weekly cases": [
                br.select(pl.corr(f, "casos", method="spearman")).item()
                for f in (NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES)
            ],
        }).with_columns(pl.col("spearman vs weekly cases").round(3))
    )
else:
    print("Run `uv run arboili transform news-lexicon` first")

national weeks with both series: 481  (2013-07-28 → 2026-08-23)

shape: (3, 2)
┌───────────────┬──────────────────────────┐
│ feature       ┆ spearman vs weekly cases │
│ ---           ┆ ---                      │
│ str           ┆ f64                      │
╞═══════════════╪══════════════════════════╡
│ news_volume   ┆ 0.347                    │
│ news_severity ┆ 0.308                    │
│ news_cases    ┆ 0.204                    │
└───────────────┴──────────────────────────┘


### Per UF

Each state gets its own coefficient, computed only over the weeks where the news
corpus names it. The `semanas` column is therefore the honest denominator: a
coefficient over 32 weeks is a different claim from one over 265, and the sort
below puts the thin ones at both ends.

In [17]:
if FEATURES_PATH.exists():
    per_uf = (
        real_features.filter(pl.col(LOCATION) != "BR")
        .join(sinan_weekly, left_on=[EW, LOCATION],
              right_on=["ew_symptom_onset", "state_abbrev"], how="inner")
        .group_by(LOCATION)
        .agg(
            pl.len().alias("semanas"),
            pl.corr(NEWS_VOLUME, "casos", method="spearman").round(3).alias("rho_volume"),
            pl.corr(NEWS_SEVERITY, "casos", method="spearman").round(3).alias("rho_severity"),
            pl.corr(NEWS_CASES, "casos", method="spearman").round(3).alias("rho_cases"),
        )
        .sort("rho_volume", descending=True)
    )
    print(per_uf.head(8))
    print(f"...\n{per_uf.height} UFs, weeks per UF {per_uf['semanas'].min()}–{per_uf['semanas'].max()}")
    print(f"median rho_volume = {per_uf['rho_volume'].median():.3f}")

shape: (8, 5)
┌──────────┬─────────┬────────────┬──────────────┬───────────┐
│ location ┆ semanas ┆ rho_volume ┆ rho_severity ┆ rho_cases │
│ ---      ┆ ---     ┆ ---        ┆ ---          ┆ ---       │
│ str      ┆ u32     ┆ f64        ┆ f64          ┆ f64       │
╞══════════╪═════════╪════════════╪══════════════╪═══════════╡
│ DF       ┆ 112     ┆ 0.496      ┆ 0.339        ┆ 0.168     │
│ SC       ┆ 53      ┆ 0.435      ┆ 0.235        ┆ 0.239     │
│ PE       ┆ 88      ┆ 0.412      ┆ 0.079        ┆ -0.183    │
│ RJ       ┆ 234     ┆ 0.381      ┆ 0.176        ┆ 0.243     │
│ SP       ┆ 265     ┆ 0.377      ┆ 0.349        ┆ 0.276     │
│ MA       ┆ 48      ┆ 0.376      ┆ -0.223       ┆ 0.106     │
│ ES       ┆ 63      ┆ 0.341      ┆ 0.055        ┆ 0.439     │
│ BA       ┆ 115     ┆ 0.327      ┆ 0.116        ┆ 0.046     │
└──────────┴─────────┴────────────┴──────────────┴───────────┘
...
27 UFs, weeks per UF 32–265
median rho_volume = 0.236


## Exploratory — lead or lag?

If media coverage carried early warning, `news_volume` at week *t* would
correlate better with cases at *t+k* than with cases at *t*. The table shifts both
series and reports the national Spearman coefficient in each direction.

In [18]:
if FEATURES_PATH.exists():
    scan = []
    for k in range(0, 9):
        shifted = br.with_columns(
            pl.col("casos").shift(-k).alias("casos_futuros"),
            pl.col(NEWS_VOLUME).shift(-k).alias("volume_futuro"),
        )
        scan.append({
            "k (weeks)": k,
            "news[t] × cases[t+k]": shifted.select(
                pl.corr(NEWS_VOLUME, "casos_futuros", method="spearman")).item(),
            "cases[t] × news[t+k]": shifted.select(
                pl.corr("casos", "volume_futuro", method="spearman")).item(),
        })
    print(pl.DataFrame(scan).with_columns(
        pl.exclude("k (weeks)").round(3)
    ))

shape: (9, 3)
┌───────────┬──────────────────────┬──────────────────────┐
│ k (weeks) ┆ news[t] × cases[t+k] ┆ cases[t] × news[t+k] │
│ ---       ┆ ---                  ┆ ---                  │
│ i64       ┆ f64                  ┆ f64                  │
╞═══════════╪══════════════════════╪══════════════════════╡
│ 0         ┆ 0.347                ┆ 0.347                │
│ 1         ┆ 0.331                ┆ 0.338                │
│ 2         ┆ 0.319                ┆ 0.317                │
│ 3         ┆ 0.306                ┆ 0.304                │
│ 4         ┆ 0.293                ┆ 0.267                │
│ 5         ┆ 0.279                ┆ 0.238                │
│ 6         ┆ 0.268                ┆ 0.21                 │
│ 7         ┆ 0.246                ┆ 0.173                │
│ 8         ┆ 0.218                ┆ 0.149                │
└───────────┴──────────────────────┴──────────────────────┘


## What the exploratory section shows

Three readings, all of them the sober kind:

- **Contemporaneous, not leading.** The coefficient peaks at `k = 0` and falls
  away in both directions, so national news volume moves *with* the epidemic curve
  rather than ahead of it. This is the empirical counterpart of the framing
  decision in [`notes/README.md`](../notes/README.md): coverage is a measure of
  attention, not an early warning.
- **Weak even so.** ~0.35 Spearman nationally is the same order as the
  counter-evidence the notes cite: Ganser 2022 found HealthMap and EIOS detected
  73.5% of outbreaks but only 9.2% within ±2 weeks.
- **`rho_cases` is not trustworthy.** It swings from strongly positive to −0.7
  across UFs because it correlates a cumulative stock against a weekly flow. It is
  shown so the reader can see it fail, and that failure is the argument for moving
  structured count extraction to braço 04.

None of this is the protocol-00 comparison. That comparison — braços 01, 02 and 03
under one model and one set of metrics — has not run, and these numbers must not
be quoted as its result.


## Running it for real

This notebook mirrors `transform()` in `src/pipeline/news/*/transform.py`:

```bash
uv run arboili transform news-text      # 4,569 articles → frozen corpus  (~45 s)
uv run arboili transform news-lexicon   # corpus → per-UF EW features     (~6 s)
```

Writes `data/pipeline/news/EBC_dengue_text.parquet` and
`..._lexicon_EW.parquet` (+ `.csv.gz`). Braços 02 (`news-embeddings`) and 03
(`news-topics`) carry their schema but refuse to run until the encoder and the
topic count are chosen.

Methods, measured facts and the seven declared limitations are in
[`src/pipeline/news/README.md`](../src/pipeline/news/README.md). The research
paths are in [`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md).

## What this notebook does not do

- **No annotation study.** Precision and recall per field are not measured here;
  there is no annotated dengue corpus in PT-BR, and building one is the open
  contribution in `notes/01`.
- **No cumulative-vs-weekly correction.** The model-side fix for repeated
  cumulative counts is an open item for protocol 00.
- **No comparison against braços 02 and 03.** They are stubs.
- **No per-UF normalisation.** The RJ-over-SP mention lead inverts around 2015,
  so a per-UF news feature is comparable to its own history before it is
  comparable across states.
